# Project 12: artemisinin- and partner-drug-resistance markers in Benin *P. falciparum* (WGS)

This notebook runs the whole analysis of Project 12 from the saved results of the mapping and calling stage.

**Before you click Save Version (Save and Run All):**

1. **Settings, Internet: ON.** It is needed to install the tools, download the reference and the SnpEff database.
2. **Add Input:** the dataset that contains `project12_results.tgz` (the archive from the 8 finished samples).
3. **Accelerator: None.** No GPU is used.

**What it does:** restores the archive, installs GATK, bcftools and SnpEff in `/tmp` (so they are not saved with the output), downloads the 3D7 reference, counts reads at every resistance-marker codon for each sample, calls each sample at the codon level as wild-type, mixed or mutant, computes prevalence with Wilson 95% confidence intervals, makes QC tables and figures, runs joint calling and SnpEff on the four loci, builds an IGV bundle, and writes a README draft.

**Expected run time:** roughly 15 to 30 minutes (tool installation is the slowest part).

**Outputs** are saved in `/kaggle/working/project12/` (`results/`, `figures/`, `README_project12.md`).

## 1. Configuration and helpers

In [ ]:
# ---------------------------------------------------------------
# Configuration and helpers
# ---------------------------------------------------------------
import os, re, sys, glob, shutil, subprocess, time, traceback
from collections import Counter
from math import sqrt
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

T0 = time.time()
P = os.environ.get("P12_WORK", "/kaggle/working/project12")      # saved with the notebook output
TOOLS = os.environ.get("P12_TOOLS", "/tmp/tools")                # conda environments, NOT saved
MM_BIN = f"{TOOLS}/bin/micromamba"
ENV_BIOINFO = f"{TOOLS}/envs/bioinfo"
ENV_SNPEFF = f"{TOOLS}/envs/snpeff"
ARCHIVE_NAME = "project12_results.tgz"
REF, GFF = f"{P}/data/ref.fa", f"{P}/data/ref.gff"
BAMDIR, GVCFDIR = f"{P}/results/bams", f"{P}/results/gvcf"
for d in ("data", "results", "logs", "figures"):
    os.makedirs(f"{P}/{d}", exist_ok=True)

# ---- analysis settings ----
ALLOW_PARTIAL = False   # True: analyse the samples that are present even if the run list is incomplete
MIN_DEPTH = 10          # fewer usable reads at a codon -> no_call
MUTANT_FRAC = 0.80      # mutant: at least this share of reads carry the mutant amino acid
MIXED_FRAC = 0.10       # mixed: share between MIXED_FRAC and MUTANT_FRAC ...
MUT_MIN_READS = 3       # ... and at least this many mutant reads
MIN_MAPQ, MIN_BASEQ = 20, 20

GENES = {"kelch13": "PF3D7_1343700", "crt": "PF3D7_0709000",
         "mdr1": "PF3D7_0523000", "dhfr": "PF3D7_0417200"}
MARKERS = {
    "kelch13": "P441L F446I G449A N458Y C469Y C469F M476I A481V Y493H R515K P527H N537I N537D G538V R539T I543T P553L R561H V568G P574L C580Y R622I A675V",
    "crt": "C72S M74I N75E K76T",
    "mdr1": "N86Y Y184F S1034C N1042D D1246Y",
    "dhfr": "A16V N51I C59R S108N S108T I164L",
}
KEY = ["C580Y", "R561H", "A675V", "K76T", "N86Y", "Y184F", "N51I", "C59R", "S108N", "I164L"]
HAP = {"crt": [72, 73, 74, 75, 76]}      # crt haplotype codons (3D7 = CVMNK, CVIET = resistant)

# ---- helpers ----
OUT = []

def say(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    OUT.append(s)

ENV = os.environ.copy()
ENV["PATH"] = f"{ENV_BIOINFO}/bin:{TOOLS}/bin:" + ENV.get("PATH", "")
ENV["MAMBA_ROOT_PREFIX"] = f"{TOOLS}/root"

def sh(cmd, check=True, show=True, cwd=None):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", cwd=cwd or P, env=ENV,
                       capture_output=True, text=True)
    if show and r.stdout.strip():
        print(r.stdout.rstrip(), flush=True)
    if r.returncode != 0:
        if check or show:
            print("COMMAND FAILED (exit %d): %s" % (r.returncode, cmd[:300]))
            print(r.stderr[-2000:])
        if check:
            raise RuntimeError("command failed, see the message above")
    return r

print("work dir:", P)
print("settings:", dict(MIN_DEPTH=MIN_DEPTH, MUTANT_FRAC=MUTANT_FRAC, MIXED_FRAC=MIXED_FRAC,
                        MUT_MIN_READS=MUT_MIN_READS, MIN_MAPQ=MIN_MAPQ, MIN_BASEQ=MIN_BASEQ))

## 2. Restore the saved results (8 samples)
Reads `project12_results.tgz` from the attached dataset: loci BAMs, gVCFs, logs and the run list.

In [ ]:
# ---------------------------------------------------------------
# Restore the saved results of the mapping and calling stage
# ---------------------------------------------------------------
cands = sorted(glob.glob(f"/kaggle/input/**/{ARCHIVE_NAME}", recursive=True))
print("archive candidates:", cands)
assert cands, ("project12_results.tgz not found under /kaggle/input. "
               "Use Add Input to attach the dataset that contains it, then run again.")
ARCH = cands[0]
print("using:", ARCH, "|", round(os.path.getsize(ARCH) / 1e6, 1), "MB")
sh(f"tar xzf '{ARCH}' -C '{P}'")

runs_todo = [l.rstrip("\n").split("\t")[0] for l in open(f"{P}/data/runs_todo.tsv") if l.strip()]
bam_samples = {os.path.basename(f)[:-len(".loci.bam")] for f in glob.glob(f"{BAMDIR}/*.loci.bam")}
gvcf_samples = {os.path.basename(f)[:-len(".g.vcf.gz")] for f in glob.glob(f"{GVCFDIR}/*.g.vcf.gz")}
samples = [s for s in runs_todo if s in bam_samples and s in gvcf_samples]
missing = [s for s in runs_todo if s not in samples]

print("\nrun list in the archive:", runs_todo)
print("samples with a loci BAM and a gVCF:", len(samples), "of", len(runs_todo))
assert samples, "no complete sample in the archive"
assert not missing or ALLOW_PARTIAL, (f"incomplete samples: {missing}. "
                                      "Fix the archive, or set ALLOW_PARTIAL = True at the top of the config cell.")

runs = pd.read_csv(f"{P}/data/runs.tsv", sep="\t")
print()
print(runs[[c for c in ["run_accession", "sample_accession", "study_accession", "collection_date",
                        "library_layout", "base_count"] if c in runs.columns]].to_string(index=False))

## 3. Install the tools
GATK, samtools, bcftools and SnpEff go to `/tmp` (not saved with the output). This is the slowest step.

In [ ]:
# ---------------------------------------------------------------
# Install the tools in /tmp (about 5 to 10 minutes)
# ---------------------------------------------------------------
os.makedirs(TOOLS, exist_ok=True)
if not os.path.exists(MM_BIN):
    sh("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba", cwd=TOOLS)
assert os.path.exists(MM_BIN), "micromamba download failed: is Internet ON in the notebook settings?"

if not os.path.exists(f"{ENV_BIOINFO}/bin/gatk"):
    r = sh(f"{MM_BIN} create -y -p {ENV_BIOINFO} -c conda-forge -c bioconda 'openjdk=17' gatk4 samtools bcftools",
           show=False)
    print(r.stdout[-300:])
if not os.path.exists(f"{ENV_SNPEFF}/bin/snpEff"):
    r = sh(f"{MM_BIN} create -y -p {ENV_SNPEFF} -c conda-forge -c bioconda snpeff", show=False)
    print(r.stdout[-300:])

try:
    import pysam
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysam"], check=True)
    import pysam

print("\ntool versions:")
sh("gatk --version 2>&1 | tail -n 3; samtools --version | head -n 1; bcftools --version | head -n 1")
sh(f"{MM_BIN} run -p {ENV_SNPEFF} snpEff -version 2>&1 | tail -n 1")
print("pysam", pysam.__version__)

## 4. Reference genome and annotation
MalariaGEN Pf7 3D7 reference and GFF.

In [ ]:
# ---------------------------------------------------------------
# Reference genome and annotation (MalariaGEN Pf7 3D7 reference)
# ---------------------------------------------------------------
def have_ref():
    return (os.path.exists(REF) and os.path.getsize(REF) > 20_000_000
            and os.path.exists(GFF) and os.path.getsize(GFF) > 15_000_000)

if not have_ref():
    for scheme in ("ftp", "https"):
        base = f"{scheme}://ngs.sanger.ac.uk/production/malaria/Resource/34"
        sh(f"wget -q --tries=3 -O {REF} {base}/Pfalciparum.genome.fasta", check=False)
        sh(f"wget -q --tries=3 -O {GFF} {base}/Pfalciparum_replace_Pf3D7_MIT_v3_with_Pf_M76611.gff", check=False)
        if have_ref():
            break
assert have_ref(), "reference download failed: is Internet ON in the notebook settings?"

sh(f"samtools faidx {REF}", show=False)
if not os.path.exists(f"{P}/data/ref.dict"):
    sh(f"gatk CreateSequenceDictionary -R {REF}", show=False)
print("reference:", round(os.path.getsize(REF) / 1e6, 1), "MB | annotation:", round(os.path.getsize(GFF) / 1e6, 1), "MB")
print(open(REF + ".fai").read().splitlines()[:3], "...")

## 5. Functions
Gene models, codon counting at the read level, Wilson interval, SnpEff parser.

In [ ]:
# ---------------------------------------------------------------
# Functions: gene models, codon counting, statistics, parsers
# ---------------------------------------------------------------
import pysam

B = "TCAG"
AA = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
CODON = {a + b + c: AA[16 * i + 4 * j + k]
         for i, a in enumerate(B) for j, b in enumerate(B) for k, c in enumerate(B)}
COMP = str.maketrans("ACGT", "TGCA")
AA3 = {"A": "Ala", "R": "Arg", "N": "Asn", "D": "Asp", "C": "Cys", "E": "Glu", "Q": "Gln",
       "G": "Gly", "H": "His", "I": "Ile", "L": "Leu", "K": "Lys", "M": "Met", "F": "Phe",
       "P": "Pro", "S": "Ser", "T": "Thr", "W": "Trp", "Y": "Tyr", "V": "Val"}


def load_models(gff_path, genes):
    # returns {gene: (chrom, strand, [genomic positions of the CDS in coding order])}
    ids = {v: k for k, v in genes.items()}
    ginfo, cds = {}, {}
    with open(gff_path) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9:
                continue
            if f[2] == "gene":
                m = re.search(r"ID=([^;]+)", f[8])
                if m and m.group(1) in ids:
                    ginfo[ids[m.group(1)]] = (f[0], f[6])
            elif f[2] == "CDS":
                m = re.search(r"Parent=([^;]+)", f[8])
                if m:
                    parent = m.group(1).split(",")[0]
                    if parent.split(".")[0] in ids:
                        cds.setdefault((ids[parent.split(".")[0]], parent), set()).add((int(f[3]), int(f[4])))
    models = {}
    for gene in genes:
        chrom, strand = ginfo[gene]
        parent = sorted(p for g, p in cds if g == gene)[0]
        exons = sorted(cds[(gene, parent)])
        pos = []
        if strand == "+":
            for s, e in exons:
                pos.extend(range(s, e + 1))
        else:
            for s, e in sorted(exons, reverse=True):
                pos.extend(range(e, s - 1, -1))
        models[gene] = (chrom, strand, pos)
    return models


def codon_positions(models, gene, n):
    chrom, strand, pos = models[gene]
    return chrom, strand, pos[3 * (n - 1): 3 * (n - 1) + 3]


def codon_ref(models, fasta, gene, n):
    chrom, strand, p = codon_positions(models, gene, n)
    cod = "".join(fasta.fetch(chrom, x - 1, x).upper() for x in p)
    return cod.translate(COMP) if strand == "-" else cod


def count_codons(bam, models, gene, codons, minmq=20, minbq=20):
    # counts, per read, the amino acids encoded at the listed codons (one string per read)
    chrom, strand, _ = models[gene]
    cps = [codon_positions(models, gene, n)[2] for n in codons]
    allpos = [x for p in cps for x in p]
    counts, seen = Counter(), set()
    for read in bam.fetch(chrom, min(allpos) - 1, max(allpos)):
        if read.is_unmapped or read.is_secondary or read.is_supplementary or read.is_duplicate:
            continue
        if read.mapping_quality < minmq or read.query_name in seen:
            continue
        m = {rp: qp for qp, rp in read.get_aligned_pairs(matches_only=True)}
        quals = read.query_qualities
        aas, ok = [], True
        for p in cps:
            if not all((x - 1) in m for x in p):
                ok = False
                break
            bases = []
            for x in p:
                qp = m[x - 1]
                if quals is not None and quals[qp] < minbq:
                    ok = False
                    break
                bases.append(read.query_sequence[qp])
            if not ok:
                break
            cod = "".join(bases)
            if strand == "-":
                cod = cod.translate(COMP)
            aas.append(CODON.get(cod, "X"))
        if ok:
            seen.add(read.query_name)
            counts["".join(aas)] += 1
    return counts


def wilson(k, n, z=1.96):
    if n == 0:
        return None, None
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half


def classify(depth, n_mut):
    frac = n_mut / depth if depth else 0.0
    if depth < MIN_DEPTH:
        return "no_call"
    if frac >= MUTANT_FRAC:
        return "mutant"
    if n_mut >= MUT_MIN_READS and frac >= MIXED_FRAC:
        return "mixed"
    return "wild-type"


def parse_ann_vcf(path, genes, markers):
    # coding changes in the four genes from a SnpEff-annotated VCF (first, most severe ANN entry)
    rev = {v: k for k, v in genes.items()}
    known = {(g, "p.%s%s%s" % (AA3[t[0]], t[1:-1], AA3[t[-1]])) for g, toks in markers.items() for t in toks.split()}
    rows = []
    with open(path) as fh:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            info = dict(kv.split("=", 1) for kv in f[7].split(";") if "=" in kv)
            ann = info.get("ANN", "").split(",")[0].split("|")
            if len(ann) < 11:
                continue
            gid = ann[4] if ann[4] in rev else ann[3]
            if gid not in rev:
                continue
            if not any(t in ann[1] for t in ("missense", "stop_gained", "frameshift", "inframe", "start_lost")):
                continue
            gene = rev[gid]
            ac = info.get("AC", "0").split(",")[0]
            rows.append({"gene": gene, "pos": int(f[1]), "ref": f[3], "alt": f[4], "effect": ann[1],
                         "protein": ann[10], "AC": int(ac) if ac.isdigit() else 0,
                         "AN": info.get("AN"), "filter": f[6], "known_marker": (gene, ann[10]) in known})
    return pd.DataFrame(rows)


def md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r.values) + " |")
    return "\n".join(lines)


def parse_flagstat(text):
    prim = int(re.search(r"(\d+) \+ \d+ primary\n", text).group(1))
    mapped = float(re.search(r"primary mapped \(([\d.]+)%", text).group(1))
    dups = int(re.search(r"(\d+) \+ \d+ primary duplicates", text).group(1))
    paired = float(re.search(r"properly paired \(([\d.]+)%", text).group(1))
    return prim, mapped, paired, round(100 * dups / prim, 2)

print("functions defined")

## 6. Gene models and marker check
Every marker's reference codon must encode the expected wild-type amino acid, otherwise the notebook stops.

In [ ]:
# ---------------------------------------------------------------
# Gene models and marker reference check
# ---------------------------------------------------------------
MODELS = load_models(GFF, GENES)
FA = pysam.FastaFile(REF)

gm = []
for gene, (chrom, strand, pos) in MODELS.items():
    prot = "".join(CODON.get(codon_ref(MODELS, FA, gene, n), "X") for n in range(1, len(pos) // 3 + 1))
    gm.append({"gene": gene, "chrom": chrom, "strand": strand, "cds_bp": len(pos),
               "protein_aa": len(prot) - (1 if prot.endswith("*") else 0),
               "internal_stops": prot[:-1].count("*"), "starts_M": prot[:1] == "M"})
print(pd.DataFrame(gm).to_string(index=False))

bad = [(g, t) for g, toks in MARKERS.items() for t in toks.split()
       if CODON.get(codon_ref(MODELS, FA, g, int(t[1:-1]))) != t[0]]
say("\nmarker reference check:", "all markers match the 3D7 reference" if not bad else f"MISMATCH {bad}")
assert not bad, "a marker number or the CDS parse is wrong"

## 7. Marker calls and prevalence
Per sample and marker: wild-type, mixed, mutant or no_call from read counts at the whole codon; prevalence with Wilson 95% intervals; crt haplotypes.

In [ ]:
# ---------------------------------------------------------------
# Marker codon counts, calls, prevalence and crt haplotypes
# ---------------------------------------------------------------
rows, hap = [], []
for s in samples:
    path = f"{BAMDIR}/{s}.loci.bam"
    if not os.path.exists(path + ".bai"):
        pysam.index(path)
    bam = pysam.AlignmentFile(path, "rb")
    for gene, toks in MARKERS.items():
        for tok in toks.split():
            wt, mut, n = tok[0], tok[-1], int(tok[1:-1])
            c = count_codons(bam, MODELS, gene, [n], MIN_MAPQ, MIN_BASEQ)
            depth = sum(c.values())
            n_wt, n_mut = c.get(wt, 0), c.get(mut, 0)
            rows.append({"sample": s, "gene": gene, "marker": tok, "depth": depth, "n_wt": n_wt,
                         "n_mut": n_mut, "n_other": depth - n_wt - n_mut,
                         "mut_frac": round(n_mut / depth, 3) if depth else 0.0,
                         "call": classify(depth, n_mut)})
    for gene, codons in HAP.items():
        ch = count_codons(bam, MODELS, gene, codons, MIN_MAPQ, MIN_BASEQ)
        tot = sum(ch.values())
        top = ch.most_common(3)
        hap.append({"sample": s, "gene": gene, "reads": tot,
                    "top_haplotype_share": round(top[0][1] / tot, 3) if tot else None,
                    "top_haplotypes": "; ".join("%s:%d" % (h, k) for h, k in top)})
    bam.close()

calls = pd.DataFrame(rows)
calls.to_csv(f"{P}/results/marker_calls.tsv", sep="\t", index=False)
haps = pd.DataFrame(hap)
haps.to_csv(f"{P}/results/haplotypes.tsv", sep="\t", index=False)

out = []
for (gene, marker), g in calls.groupby(["gene", "marker"], sort=False):
    called = g[g["call"] != "no_call"]
    n = len(called)
    k_mut = int((called["call"] == "mutant").sum())
    k_mix = int((called["call"] == "mixed").sum())
    lo, hi = wilson(k_mut + k_mix, n)
    out.append({"gene": gene, "marker": marker, "n_called": n, "wild_type": n - k_mut - k_mix,
                "mixed": k_mix, "mutant": k_mut,
                "any_mutant_pct": round(100 * (k_mut + k_mix) / n, 1) if n else None,
                "ci_low": round(100 * lo, 1) if lo is not None else None,
                "ci_high": round(100 * hi, 1) if hi is not None else None,
                "median_depth": float(g["depth"].median())})
res = pd.DataFrame(out)
res.to_csv(f"{P}/results/marker_prevalence.tsv", sep="\t", index=False)

wide = calls[calls["marker"].isin(KEY)].pivot(index="sample", columns="marker", values="call")
wide = wide[[m for m in KEY if m in wide.columns]]
wide.to_csv(f"{P}/results/key_marker_calls_wide.tsv", sep="\t")

say("== marker prevalence (key markers, plus any marker with a mutant or mixed call) ==")
say(res[(res["mixed"] + res["mutant"] > 0) | res["marker"].isin(KEY)].to_string(index=False))
say("\nmedian codon depth per sample:", calls.groupby("sample")["depth"].median().to_dict())
say("\nhaplotypes at the crt codons %d-%d (3D7 = CVMNK, CVIET = resistant at codons 72-76):" % (HAP["crt"][0], HAP["crt"][-1]))
say(haps.to_string(index=False))
say("\nkey-marker calls per sample (wild-type / mixed / mutant / no_call):")
say(wide.to_string())
hits = calls[calls["marker"].isin(KEY) & calls["call"].isin(["mutant", "mixed"])]
say("\nkey-marker mutant or mixed calls:")
say(hits[["sample", "gene", "marker", "depth", "n_mut", "mut_frac", "call"]].to_string(index=False)
    if len(hits) else "none")

## 8. QC

In [ ]:
# ---------------------------------------------------------------
# QC table
# ---------------------------------------------------------------
qc = []
for s in samples:
    fp = f"{P}/logs/flagstat_{s}.txt"
    row = {"sample": s}
    if os.path.exists(fp):
        prim, mapped, paired, dup = parse_flagstat(open(fp).read())
        row.update({"primary_reads": prim, "mapped_pct": mapped, "properly_paired_pct": paired, "dup_pct": dup})
    row["median_codon_depth"] = float(calls.loc[calls["sample"] == s, "depth"].median())
    qc.append(row)
qc = pd.DataFrame(qc)
qc.to_csv(f"{P}/results/qc.tsv", sep="\t", index=False)
say("== QC ==")
say(qc.to_string(index=False))
low = qc[qc["median_codon_depth"] < 20]
if len(low):
    say("\nWARNING: samples with median codon depth below 20:", low["sample"].tolist())

## 9. Figures

In [ ]:
# ---------------------------------------------------------------
# Figures: key-marker heatmap and prevalence with 95% confidence intervals
# ---------------------------------------------------------------
try:
    import numpy as np
    import matplotlib.pyplot as plt

    # 1. mutant-read fraction per sample and key marker
    sub = calls[calls["marker"].isin(KEY)]
    cols = [m for m in KEY if m in set(sub["marker"])]
    frac = sub.pivot(index="sample", columns="marker", values="mut_frac")[cols]
    call = sub.pivot(index="sample", columns="marker", values="call")[cols]
    mat = np.ma.masked_where((call == "no_call").values, frac.values.astype(float))
    fig, ax = plt.subplots(figsize=(max(7.5, 2.0 + 0.85 * len(cols)), max(3.2, 1.4 + 0.5 * len(frac))))
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=45, ha="right")
    ax.set_yticks(range(len(frac)))
    ax.set_yticklabels(frac.index)
    for i in range(frac.shape[0]):
        for j in range(frac.shape[1]):
            txt = "nc" if call.values[i, j] == "no_call" else "%.2f" % frac.values[i, j]
            ax.text(j, i, txt, ha="center", va="center", fontsize=8,
                    color="white" if (txt != "nc" and frac.values[i, j] < 0.6) else "black")
    fig.colorbar(im, ax=ax, label="mutant read fraction")
    ax.set_title("Mutant read fraction per sample (nc = under %d reads)" % MIN_DEPTH, fontsize=10)
    fig.tight_layout()
    fig.savefig(f"{P}/figures/key_marker_heatmap.png", dpi=150)
    plt.show()

    # 2. prevalence (mutant or mixed) with Wilson 95% intervals
    r = res[res["marker"].isin(KEY) & (res["n_called"] > 0)].copy()
    r["label"] = r["gene"] + " " + r["marker"] + " (n=" + r["n_called"].astype(int).astype(str) + ")"
    r = r.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(r) + 1.8))
    x = r["any_mutant_pct"].astype(float).values
    ax.errorbar(x, range(len(r)),
                xerr=[x - r["ci_low"].astype(float).values, r["ci_high"].astype(float).values - x],
                fmt="o", capsize=3)
    ax.set_yticks(range(len(r)))
    ax.set_yticklabels(r["label"])
    ax.set_xlim(0, 100)
    ax.set_xlabel("samples with a mutant or mixed call (%), Wilson 95% CI")
    ax.set_title("Resistance markers: prevalence (Benin, PRJEB96783)", fontsize=10)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"{P}/figures/marker_prevalence.png", dpi=150)
    plt.show()
    say("figures saved in", f"{P}/figures")
except Exception as e:
    traceback.print_exc()
    say("SECTION FAILED (figures):", e)

## 10. Joint calling and SnpEff on the four loci
Discovery side: every coding change at the four genes, to cross-check the read counts.

In [ ]:
# ---------------------------------------------------------------
# Joint calling on the four loci, SnpEff annotation, coding-variant table
# ---------------------------------------------------------------
disc = pd.DataFrame()
try:
    say("== joint calling and annotation ==")
    for s in samples:
        g = f"{GVCFDIR}/{s}.g.vcf.gz"
        if not os.path.exists(g + ".tbi"):
            sh(f"bcftools index -t -f '{g}'", show=False)
    args = " ".join(f"--variant {GVCFDIR}/{s}.g.vcf.gz" for s in samples)
    sh(f"gatk CombineGVCFs -R {REF} {args} -O results/cohort.g.vcf.gz 2> logs/combine.log", show=False)
    sh(f"gatk GenotypeGVCFs -R {REF} -V results/cohort.g.vcf.gz -O results/cohort.vcf.gz "
       "--sample-ploidy 1 -L data/loci.bed 2> logs/genotype.log", show=False)
    sh(f"gatk VariantFiltration -R {REF} -V results/cohort.vcf.gz -O results/cohort.filtered.vcf.gz "
       '--filter-expression "QD < 2.0" --filter-name QD2 --filter-expression "FS > 60.0" --filter-name FS60 '
       '--filter-expression "SOR > 3.0" --filter-name SOR3 --filter-expression "MQ < 40.0" --filter-name MQ40 '
       "2> logs/filter.log", show=False)

    # SnpEff database uses the chromosome names 1-14, the reference uses Pf3D7_NN_v3
    with open(f"{P}/results/rename_to_db.txt", "w") as a, open(f"{P}/results/rename_back.txt", "w") as b:
        for i in range(1, 15):
            a.write("Pf3D7_%02d_v3\t%d\n" % (i, i))
            b.write("%d\tPf3D7_%02d_v3\n" % (i, i))
    SNPEFF = f"{MM_BIN} run -p {ENV_SNPEFF} snpEff"
    if not glob.glob(f"{ENV_SNPEFF}/share/snpeff-*/data/Plasmodium_falciparum/snpEffectPredictor.bin"):
        sh(f"{SNPEFF} download -v Plasmodium_falciparum 2>&1 | tail -n 2")
    sh("bcftools annotate --rename-chrs results/rename_to_db.txt -O z "
       "-o results/cohort.renamed.vcf.gz results/cohort.filtered.vcf.gz", show=False)
    sh(f"{SNPEFF} -v -stats results/loci.snpeff.html Plasmodium_falciparum results/cohort.renamed.vcf.gz "
       "> results/cohort.ann.renamed.vcf 2> logs/snpeff.log", show=False)
    sh("bcftools annotate --rename-chrs results/rename_back.txt -o results/cohort.ann.vcf "
       "results/cohort.ann.renamed.vcf", show=False)

    ann_vcf = f"{P}/results/cohort.ann.vcf"
    txt = open(ann_vcf).read()
    n_rec = sum(1 for l in txt.splitlines() if l and not l.startswith("#"))
    say("records:", n_rec, "| ERROR_CHROMOSOME_NOT_FOUND:", txt.count("ERROR_CHROMOSOME_NOT_FOUND"),
        "| WARNING_REF_DOES_NOT_MATCH_GENOME:", txt.count("WARNING_REF_DOES_NOT_MATCH_GENOME"))

    disc = parse_ann_vcf(ann_vcf, GENES, MARKERS)
    if len(disc):
        disc = disc.sort_values(["gene", "pos"])
        disc.to_csv(f"{P}/results/locus_variants.tsv", sep="\t", index=False)
        say("\ncoding changes per gene (known markers = the markers counted above):")
        say(disc.groupby("gene").agg(variants=("pos", "count"), known_markers=("known_marker", "sum")).to_string())
        km = disc[disc["known_marker"]]
        say("\nknown markers in the joint call (AC = samples carrying the alternate allele, AN = samples genotyped):")
        say(km.drop(columns="known_marker").to_string(index=False) if len(km) else "none")
        say("\nother non-synonymous changes with the highest AC (all in results/locus_variants.tsv):")
        say(disc[~disc["known_marker"]].sort_values("AC", ascending=False).head(25)
            .drop(columns="known_marker").to_string(index=False))
    else:
        say("no coding changes found: check logs/snpeff.log")
except Exception as e:
    traceback.print_exc()
    say("SECTION FAILED (joint calling / SnpEff):", e)

## 11. IGV bundle
Download `results/igv_bundle` from the notebook output and open it in IGV on your computer for the screenshots.

In [ ]:
# ---------------------------------------------------------------
# IGV bundle (open on your computer: reference, loci BAMs, gene models, marker locations)
# ---------------------------------------------------------------
try:
    bdir = f"{P}/results/igv_bundle"
    os.makedirs(bdir, exist_ok=True)
    for s in samples:
        for ext in (".loci.bam", ".loci.bam.bai"):
            if os.path.exists(f"{BAMDIR}/{s}{ext}"):
                shutil.copy(f"{BAMDIR}/{s}{ext}", bdir)
    shutil.copy(REF, bdir)
    shutil.copy(REF + ".fai", bdir)
    bed = [l.rstrip("\n").split("\t")[:3] for l in open(f"{P}/data/loci.bed") if l.strip()]
    with open(f"{bdir}/loci.gff3", "w") as g_out, open(GFF) as g_in:
        g_out.write("##gff-version 3\n")
        for line in g_in:
            f = line.split("\t")
            if line.startswith("#") or len(f) < 9:
                continue
            if any(f[0] == c and int(f[3]) <= int(e) and int(f[4]) > int(st) for c, st, e in bed):
                g_out.write(line)
    with open(f"{bdir}/igv_locations.txt", "w") as o:
        o.write("gene\tmarker\tlocation\n")
        for gene, toks in MARKERS.items():
            for tok in toks.split():
                n = int(tok[1:-1])
                mid = sorted(codon_positions(MODELS, gene, n)[2])[1]
                o.write("%s\t%s\t%s:%d-%d\n" % (gene, tok, MODELS[gene][0], mid - 20, mid + 20))
    say("IGV bundle:", bdir)
    say(sh(f"du -sh {bdir}", show=False).stdout.strip())
    say("first locations:")
    say("".join(open(f"{bdir}/igv_locations.txt").readlines()[:6]).rstrip())
except Exception as e:
    traceback.print_exc()
    say("SECTION FAILED (IGV bundle):", e)

## 12. README draft and summary

In [ ]:
# ---------------------------------------------------------------
# README draft, summary file, final listing
# ---------------------------------------------------------------
def versions():
    cmds = {"GATK": "gatk --version 2>&1 | grep -i 'Genome Analysis' ",
            "samtools": "samtools --version | head -n 1", "bcftools": "bcftools --version | head -n 1",
            "SnpEff": f"{MM_BIN} run -p {ENV_SNPEFF} snpEff -version 2>&1 | tail -n 1"}
    out = []
    for k, c in cmds.items():
        try:
            txt = sh(c, check=False, show=False).stdout.strip()
            if txt and "No such file" not in txt and "not found" not in txt:
                out.append("%s: %s" % (k, txt.splitlines()[-1]))
        except Exception:
            pass
    return "; ".join(out) if out else "see the notebook output"

runs_t = pd.read_csv(f"{P}/data/runs.tsv", sep="\t")
runs_t = runs_t[runs_t["run_accession"].isin(samples)]
tab_runs = runs_t[[c for c in ["run_accession", "sample_accession", "study_accession", "collection_date"]
                   if c in runs_t.columns]].copy()
if "base_count" in runs_t.columns:
    tab_runs["yield_Gb"] = (runs_t["base_count"].astype(float) / 1e9).round(2).values

keyres = res[res["marker"].isin(KEY)][["gene", "marker", "n_called", "wild_type", "mixed", "mutant",
                                        "any_mutant_pct", "ci_low", "ci_high", "median_depth"]]
detected = res[(res["mixed"] + res["mutant"] > 0) & ~res["marker"].isin(KEY)][
    ["gene", "marker", "n_called", "wild_type", "mixed", "mutant", "any_mutant_pct", "ci_low", "ci_high"]]

L = []
L.append("# Project 12: resistance markers in Benin *P. falciparum* (WGS)\n")
L.append("Igbega X Bioinformatics Self-Training Program, Saturday project of Week 12. Draft generated by the notebook; "
         "the interpretation section must be completed after review.\n")
L.append("## Objective\n")
L.append("Estimate the prevalence of artemisinin-resistance (kelch13) and partner-drug or antifolate markers "
         "(crt, mdr1, dhfr) in *P. falciparum* from Benin using whole-genome sequencing data from ENA, "
         "classifying each sample as wild-type, mixed or mutant at every marker codon.\n")
L.append("## Samples (%d runs, one study)\n" % len(samples))
L.append(md_table(tab_runs) + "\n")
L.append("The runs were drawn at random from the Benin runs of 1.0 to 2.5 Gb. Prevalence estimates describe "
         "these parasites (one study, collection years in the table), not Benin in general.\n")
L.append("## Methods\n")
L.append("- Reads capped to 5 million pairs per sample (seqtk, seed 42), mapped with bwa mem to the MalariaGEN Pf7 "
         "3D7 reference, duplicates marked (GATK MarkDuplicates); loci BAMs hold reads overlapping each gene +/- 2 kb. "
         "This stage ran in an earlier session; this notebook restores its saved output.")
L.append("- HaplotypeCaller (haploid, gVCF) on the four loci; CombineGVCFs and GenotypeGVCFs; hard filters "
         "(QD < 2, FS > 60, SOR > 3, MQ < 40); SnpEff database Plasmodium_falciparum "
         "(contigs renamed 1-14 and back).")
L.append("- Marker calls come from read counts at the whole codon: reads with MAPQ >= %d, base quality >= %d at "
         "the three bases, not duplicates, one count per read pair. no_call if fewer than %d reads; mutant if "
         "at least %d%% of reads carry the mutant amino acid; mixed if at least %d mutant reads and at least "
         "%d%%; otherwise wild-type. Prevalence = samples with a mutant or mixed call among called samples, "
         "Wilson 95%% confidence interval." % (MIN_MAPQ, MIN_BASEQ, MIN_DEPTH, int(MUTANT_FRAC * 100),
                                              MUT_MIN_READS, int(MIXED_FRAC * 100)))
L.append("- Versions: " + versions() + "\n")
L.append("## QC\n")
L.append(md_table(qc) + "\n")
L.append("## Results\n")
L.append("### Key markers\n")
L.append(md_table(keyres) + "\n")
if len(detected):
    L.append("### Other markers with a mutant or mixed call\n")
    L.append(md_table(detected) + "\n")
else:
    L.append("No other panel marker had a mutant or mixed call.\n")
L.append("### crt haplotype at codons %d-%d (3D7 = CVMNK, CVIET = resistant at codons 72-76)\n" % (HAP["crt"][0], HAP["crt"][-1]))
L.append(md_table(haps) + "\n")
if len(disc):
    L.append("### Coding changes found by joint calling and SnpEff\n")
    L.append(md_table(disc.groupby("gene").agg(variants=("pos", "count"),
                                               known_markers=("known_marker", "sum")).reset_index()) + "\n")
    L.append("The full list is in results/locus_variants.tsv.\n")
L.append("Figures: figures/key_marker_heatmap.png, figures/marker_prevalence.png.\n")
L.append("## Interpretation\n")
L.append("TO COMPLETE after reviewing the tables above.\n")
L.append("## Limitations\n")
L.append("- %d samples from one study give wide confidence intervals: 0 of %d is compatible with a true "
         "prevalence up to about %d%%." % (len(samples), len(samples), round(100 * wilson(0, len(samples))[1])))
L.append("- Reads were capped at 5 million pairs per sample; minor clones below about 10% of reads are not detectable.")
L.append("- A haploid genotype call hides mixed infections; the marker calls above use read counts instead.")
L.append("- No copy-number analysis (for example mdr1 amplification) was done.\n")
L.append("## Files\n")
L.append("- results/: marker_calls.tsv, marker_prevalence.tsv, key_marker_calls_wide.tsv, haplotypes.tsv, qc.tsv, "
         "locus_variants.tsv, cohort VCFs, SnpEff stats, igv_bundle/")
L.append("- figures/: key_marker_heatmap.png, marker_prevalence.png")
L.append("- data/runs.tsv: the runs analysed\n")
open(f"{P}/README_project12.md", "w").write("\n".join(L))

say("\nelapsed minutes:", round((time.time() - T0) / 60, 1))
with open(f"{P}/results/project12_summary.txt", "w") as fh:
    fh.write("\n".join(OUT) + "\n")
sh(f"cd {P} && tar czf project12_igv_bundle.tgz -C results igv_bundle 2>/dev/null; "
   "find . -maxdepth 2 -type f -not -path './results/bams/*' -not -path './results/gvcf/*' "
   "-not -path './results/igv_bundle/*' | sort | head -n 60")
print("\nREADME draft:", f"{P}/README_project12.md")